# Synthetic athlete-monitoring data: leakage-controlled Gaussian evaluation, privacy proxies and sensitivity analysis

This notebook reproduces the analyses added to the manuscript *Synthetic Athlete-Monitoring Data Generation: Statistical Fidelity, Predictive Utility and Distance-Based Privacy Proxies*, using the two datasets supplied (`UConnCondensedMasterDatasetReal__3_.csv` = Plantiga activity data; `RSI_Final.csv` = SHU/WHOOP data).

**Protocol (as described in the manuscript):** athlete/person-grouped 75:25 splits (`GroupShuffleSplit`, seeds 0-9); median imputation fitted on the training partition only; multivariate Gaussian generator (standardised features, Ledoit-Wolf covariance, dispersion factor alpha, clipping to training min/max); Random Forest (50 trees, depth 10, all features, `random_state=42`); TSTR evaluated on untouched real test records (observed RSI only for SHU).

**Notes:** the exact seed list and predictor set of the original runs were not available, so these results are an independent re-implementation. They reproduce the manuscript values closely (e.g. Plantiga TRTR 0.852 vs 0.843; SHU TRTR 0.303 vs 0.304) but should be replaced by your own run where they differ. CTGAN is not run here (original code/data unavailable).

In [1]:
import os, re, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from scipy.stats import wasserstein_distance
from sklearn.ensemble import RandomForestRegressor
from sklearn.covariance import LedoitWolf
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, roc_auc_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
pd.set_option('display.width',200)
DATA_DIR = '/mnt/user-data/uploads/' if os.path.exists('/mnt/user-data/uploads/RSI_Final.csv') else './'
PLANTIGA_CSV = DATA_DIR + 'UConnCondensedMasterDatasetReal__3_.csv'
SHU_CSV = DATA_DIR + 'RSI_Final.csv'
SEEDS = list(range(10)); ALPHAS = [0.6, 0.8, 1.0, 1.2]

## 1. Load and clean the datasets

In [2]:
# ---- SHU (WHOOP) dataset
w = pd.read_csv(SHU_CSV)
w['Week'] = w['Week'].str.strip().str.capitalize()            # 'week 7' and 'Week 7' are the same week
for c in ('Monotony','Strain'):                                  # '#DIV/0!' spreadsheet errors -> missing
    w[c] = pd.to_numeric(w[c], errors='coerce')
shu_num = [c for c in w.columns if c not in ('Athlete','Week','RSI_Level')]
# ---- Plantiga dataset (long metric table -> one row per activity)
u = pd.read_csv(PLANTIGA_CSV)
p = u.pivot_table(index='Activity ID', columns='Metric Name', values='Value', aggfunc='first')
meta = u.drop_duplicates('Activity ID').set_index('Activity ID')[['Time','Person ID']]
p = p.join(meta); p['Time'] = pd.to_datetime(p['Time'], utc=True, format='ISO8601')
p['Week'] = ((p['Time'] - p['Time'].min()).dt.days // 7) + 1
pl_num = [c for c in p.columns if c not in ('Time','Person ID','Week')]
print('SHU:', w.shape, '| athletes:', w.Athlete.nunique(), '| numeric variables:', len(shu_num))
print('Plantiga:', p.shape[0], 'activities |', p['Person ID'].nunique(), 'athletes |', len(pl_num), 'numeric variables')

SHU: (494, 42) | athletes: 17 | numeric variables: 39
Plantiga: 493 activities | 7 athletes | 14 numeric variables


## 2. Data verification checks (facts quoted in the manuscript)

In [3]:
print('Distinct weeks after harmonising labels:', w.Week.nunique())
print('Rows per athlete (SHU):'); print(w.groupby('Athlete').size().to_string())
print('\nMissing cells (SHU):', int(w.isna().sum().sum()), 'of', w.size, f'= {w.isna().sum().sum()/w.size*100:.2f}%')
m = w.isna().sum(); print((m[m>0]).to_string())
print(f"\nRSI missing: {w.RSI.isna().sum()} ({w.RSI.isna().mean()*100:.2f}%), observed n={w.RSI.notna().sum()}, mean={w.RSI.mean():.4f}, SD={w.RSI.std():.4f}, median={w.RSI.median():.2f}, range={w.RSI.min()}-{w.RSI.max()}")
print('Observed Body.Weight mean %.2f | Peak.Power mean %.2f' % (w['Body.Weight'].mean(), w['Peak.Power'].mean()))
print('\nMonotony==0 rows:', int((w.Monotony==0).sum()), '| Weekly.SD==0 rows:', int((w['Weekly.SD']==0).sum()),
      '| Monotony==0 & Weekly.SD==0:', int(((w.Monotony==0)&(w['Weekly.SD']==0)).sum()), '| #DIV/0! (now NaN):', int(w.Monotony.isna().sum()))
print('Sleep > Bed violations in real data:', int((w['Hours.of.Sleep']>w['Hours.in.Bed']).sum()))
r = (w['Daily.Average']/w['Week.Trimp.Total']).replace([np.inf,-np.inf],np.nan).dropna()
print('\nDaily.Average / Week.Trimp.Total takes values (1/k):'); print(r.round(3).value_counts().head(8).to_string())
print('\nPlantiga missing cells:', int(p[pl_num].isna().sum().sum()), 'of', p[pl_num].size)
a, b = p['Athlete Intensity Asymmetry'], p['Total Athlete g-Load Asymmetry']
print('Identical asymmetry columns (rtol 1e-6):', int(np.isclose(a,b,rtol=1e-6).sum()), 'of', int(a.notna().sum()))
ratio = p['Total Athlete g-Load']/(p['Athlete Intensity']*(p['Move Time']+p['Idle Time']))
print('Total g-Load / (Intensity x (Move+Idle)): median %.4f (n=%d)' % (ratio.median(), ratio.notna().sum()))

Distinct weeks after harmonising labels: 26
Rows per athlete (SHU):
Athlete
Athlete 1 Bergdorf          26
Athlete 10 McCray           26
Athlete 11 Owusu-Mensah     26
Athlete 12 Porter           26
Athlete 13 Smith            26
Athlete 14 Stroemel         26
Athlete 15 Tucker           26
Athlete 16 Wood             26
Athlete 17 Laeticia Ziba    26
Athlete 2 Hagood            78
Athlete 3 Hammond           26
Athlete 4 Hallum            26
Athlete 5 Jackson           26
Athlete 6 Johnson           26
Athlete 7 Klein             26
Athlete 8 Marshall          26
Athlete 9 Martino           26

Missing cells (SHU): 999 of 20748 = 4.81%
Monotony            3
Strain              3
RT.Volume.Load      8
Body.Weight       144
Peak.Power        145
Peak.Power.BM     145
RSI               203
Jump.Height       145
RSI_Level         203

RSI missing: 203 (41.09%), observed n=291, mean=0.3716, SD=0.0765, median=0.36, range=0.2-0.67
Observed Body.Weight mean 74.87 | Peak.Power mean 3328.46

M

## 3. Gaussian generator and evaluation functions

In [4]:
def mvg(Xtr, alpha, rng):
    """Median-impute, standardise, Ledoit-Wolf covariance, sample N(0, alpha^2 S), invert scaling, clip to training range."""
    med = np.nanmedian(Xtr, axis=0); X = np.where(np.isnan(Xtr), med, Xtr)
    mu, sd = X.mean(0), X.std(0); sd[sd==0] = 1
    Z = (X-mu)/sd; S = LedoitWolf().fit(Z).covariance_
    Zs = rng.multivariate_normal(np.zeros(Z.shape[1]), alpha**2*S, size=len(Z), check_valid='ignore')
    return np.clip(Zs*sd+mu, X.min(0), X.max(0)), X

def rf(): return RandomForestRegressor(n_estimators=50, max_depth=10, max_features=None, random_state=42)
def met(y, p): return dict(r2=r2_score(y,p), rmse=float(np.sqrt(mean_squared_error(y,p))), mae=mean_absolute_error(y,p))

CFG = {
 'Plantiga': dict(X=p[pl_num].values.astype(float), g=p['Person ID'].values, cols=pl_num, target='Total Athlete g-Load',
                  pred=['Move Time','Idle Time','Athlete Intensity','Average Impact','Average Push-off','Average Impact Asymmetry','Average Push-off Asymmetry']),
 'SHU': dict(X=w[shu_num].values.astype(float), g=w['Athlete'].values, cols=shu_num, target='RSI',
             pred=[c for c in shu_num if c not in ('RSI','Jump.Height')])}   # RSI_Level is categorical and not in the numeric table

def split(cfg, s):
    return next(GroupShuffleSplit(1, test_size=0.25, random_state=s).split(cfg['X'], groups=cfg['g']))

def utility(name, alpha, seeds=SEEDS):
    cfg = CFG[name]; X, cols = cfg['X'], cfg['cols']; ti = cols.index(cfg['target']); pi = [cols.index(c) for c in cfg['pred']]
    rows = []
    for s in seeds:
        tr, te = split(cfg, s); rng = np.random.default_rng(s)
        med = np.nanmedian(X[tr], axis=0); Xtr = np.where(np.isnan(X[tr]), med, X[tr]); Xte = np.where(np.isnan(X[te]), med, X[te])
        obs_tr = ~np.isnan(X[tr][:,ti]); obs_te = ~np.isnan(X[te][:,ti])          # observed targets only (matters for SHU)
        real = rf().fit(Xtr[obs_tr][:,pi], X[tr][obs_tr,ti])
        trtr = met(X[te][obs_te,ti], real.predict(Xte[obs_te][:,pi]))
        syn, Xi = mvg(X[tr], alpha, rng)
        tstr = met(X[te][obs_te,ti], rf().fit(syn[:,pi], syn[:,ti]).predict(Xte[obs_te][:,pi]))
        sw = np.mean([wasserstein_distance(Xi[:,j], syn[:,j])/Xi[:,j].std() for j in range(X.shape[1]) if Xi[:,j].std()>0])
        iu = np.triu_indices(X.shape[1], 1)
        cd = np.nanmean(np.abs(np.corrcoef(Xi.T)-np.nan_to_num(np.corrcoef(syn.T)))[iu])
        rows.append(dict(seed=s, **{'TRTR_'+k:v for k,v in trtr.items()}, **{'TSTR_'+k:v for k,v in tstr.items()}, std_W=sw, corr_diff=cd))
    return pd.DataFrame(rows)
print('functions defined')

functions defined


## 4. Leakage-controlled predictive utility (TRTR and TSTR), alpha = 0.8

In [5]:
main = {}
for name in CFG:
    df = utility(name, 0.8); main[name] = df
    print(f'--- {name}: target = {CFG[name]["target"]}; predictors = {CFG[name]["pred"] if name=="Plantiga" else str(len(CFG[name]["pred"]))+" numeric variables"}')
    print(df.drop(columns='seed').agg(['mean','std']).round(3).T.to_string()); print()

--- Plantiga: target = Total Athlete g-Load; predictors = ['Move Time', 'Idle Time', 'Athlete Intensity', 'Average Impact', 'Average Push-off', 'Average Impact Asymmetry', 'Average Push-off Asymmetry']
                mean       std
TRTR_r2        0.852     0.061
TRTR_rmse  17707.834  4323.373
TRTR_mae   12470.140  3155.936
TSTR_r2        0.662     0.153
TSTR_rmse  26321.647  4224.233
TSTR_mae   20191.923  3949.430
std_W          0.182     0.017
corr_diff      0.036     0.013

--- SHU: target = RSI; predictors = 37 numeric variables
            mean    std
TRTR_r2    0.303  0.245
TRTR_rmse  0.054  0.012
TRTR_mae   0.040  0.011
TSTR_r2    0.374  0.191
TSTR_rmse  0.053  0.018
TSTR_mae   0.039  0.011
std_W      0.206  0.007
corr_diff  0.042  0.004



## 5. Sensitivity to the dispersion factor alpha (Tables 12-13 of the manuscript)

In [6]:
sens = []
for name in CFG:
    for a_ in ALPHAS:
        df = utility(name, a_)
        sens.append(dict(dataset=name, alpha=a_, std_W=f'{df.std_W.mean():.3f} +/- {df.std_W.std():.3f}', corr_diff=f'{df.corr_diff.mean():.3f} +/- {df.corr_diff.std():.3f}',
                         TRTR_R2=f'{df.TRTR_r2.mean():.3f} +/- {df.TRTR_r2.std():.3f}', TSTR_R2=f'{df.TSTR_r2.mean():.3f} +/- {df.TSTR_r2.std():.3f}'))
sens = pd.DataFrame(sens); print(sens.to_string(index=False))

 dataset  alpha           std_W       corr_diff         TRTR_R2         TSTR_R2
Plantiga    0.6 0.327 +/- 0.012 0.036 +/- 0.013 0.852 +/- 0.061 0.597 +/- 0.202
Plantiga    0.8 0.182 +/- 0.017 0.036 +/- 0.013 0.852 +/- 0.061 0.662 +/- 0.153
Plantiga    1.0 0.116 +/- 0.021 0.036 +/- 0.013 0.852 +/- 0.061 0.694 +/- 0.129
Plantiga    1.2 0.189 +/- 0.029 0.037 +/- 0.012 0.852 +/- 0.061 0.693 +/- 0.112
     SHU    0.6 0.293 +/- 0.007 0.042 +/- 0.004 0.303 +/- 0.245 0.340 +/- 0.200
     SHU    0.8 0.206 +/- 0.007 0.042 +/- 0.004 0.303 +/- 0.245 0.374 +/- 0.191
     SHU    1.0 0.198 +/- 0.007 0.042 +/- 0.004 0.303 +/- 0.245 0.366 +/- 0.205
     SHU    1.2 0.279 +/- 0.010 0.042 +/- 0.004 0.303 +/- 0.245 0.349 +/- 0.217


## 6. Distance-based privacy proxies (Tables 10-11)
Distances are Euclidean distances in standardised feature space divided by sqrt(number of features), so they are comparable across datasets. NNDR = distance to nearest / second-nearest training record. Membership-inference score = negative distance of a real record to its nearest synthetic record (members = training rows, non-members = athlete-held-out rows). Because non-members are *different athletes*, an AUC above 0.5 mainly reflects athlete-level similarity.

In [7]:
def privacy(name, alpha, seeds=SEEDS):
    cfg = CFG[name]; X = cfg['X']; rows = []
    for s in seeds:
        tr, te = split(cfg, s); rng = np.random.default_rng(s)
        syn, Xtr = mvg(X[tr], alpha, rng); med = np.nanmedian(X[tr], axis=0); Xte = np.where(np.isnan(X[te]), med, X[te])
        mu, sd = Xtr.mean(0), Xtr.std(0); sd[sd==0] = 1; f = lambda A: (A-mu)/sd; k = np.sqrt(X.shape[1])
        A, B, C = f(Xtr), f(Xte), f(syn)
        nn = NearestNeighbors(n_neighbors=2).fit(A); ds, _ = nn.kneighbors(C); dh, _ = nn.kneighbors(B)
        nns = NearestNeighbors(n_neighbors=1).fit(C); dm = nns.kneighbors(A)[0][:,0]; dn = nns.kneighbors(B)[0][:,0]
        auc = roc_auc_score(np.r_[np.ones(len(dm)), np.zeros(len(dn))], -np.r_[dm, dn])
        dup = int((np.round(syn,8)[:,None,:]==np.round(Xtr,8)[None,:,:]).all(2).any(1).sum()) if len(syn)*len(Xtr)<4e6 else -1
        rows.append(dict(DCR_syn=np.median(ds[:,0])/k, DCR_heldout_to_syn=np.median(dn)/k, NNDR_syn=np.median(ds[:,0]/np.maximum(ds[:,1],1e-12)),
                         NNDR_heldout=np.median(dh[:,0]/np.maximum(dh[:,1],1e-12)), exact_duplicates=dup, MIA_AUC=auc))
    return pd.DataFrame(rows)
priv = []
for name in CFG:
    for a_ in ALPHAS:
        d_ = privacy(name, a_); row = {'dataset':name, 'alpha':a_}
        for c in d_.columns: row[c] = f'{d_[c].mean():.3f} +/- {d_[c].std():.3f}' if c!='exact_duplicates' else int(d_[c].sum())
        priv.append(row)
priv = pd.DataFrame(priv); print(priv.to_string(index=False))

 dataset  alpha         DCR_syn DCR_heldout_to_syn        NNDR_syn    NNDR_heldout  exact_duplicates         MIA_AUC
Plantiga    0.6 0.331 +/- 0.005    0.489 +/- 0.078 0.912 +/- 0.006 0.917 +/- 0.010                 0 0.633 +/- 0.094
Plantiga    0.8 0.380 +/- 0.006    0.467 +/- 0.054 0.917 +/- 0.006 0.917 +/- 0.010                 0 0.639 +/- 0.082
Plantiga    1.0 0.445 +/- 0.010    0.473 +/- 0.042 0.925 +/- 0.007 0.917 +/- 0.010                 0 0.634 +/- 0.067
Plantiga    1.2 0.521 +/- 0.009    0.499 +/- 0.029 0.930 +/- 0.005 0.917 +/- 0.010                 0 0.623 +/- 0.051
     SHU    0.6 0.536 +/- 0.007    0.769 +/- 0.079 0.978 +/- 0.015 0.941 +/- 0.029                 0 0.559 +/- 0.073
     SHU    0.8 0.663 +/- 0.007    0.783 +/- 0.068 0.977 +/- 0.014 0.941 +/- 0.029                 0 0.567 +/- 0.072
     SHU    1.0 0.785 +/- 0.010    0.823 +/- 0.057 0.976 +/- 0.010 0.941 +/- 0.029                 0 0.571 +/- 0.071
     SHU    1.2 0.899 +/- 0.010    0.881 +/- 0.049 0.973 +/- 0.0

## 7. Supporting checks: imputation quality by masking observed values (RF-MICE vs median)

In [8]:
rng = np.random.default_rng(100); cols = ['RSI','Peak.Power','Peak.Power.BM','Jump.Height','Body.Weight','RT.Volume.Load']
X0 = w[shu_num].copy(); truth = {}
for c in cols:
    idx = X0.index[X0[c].notna()]; m_ = rng.choice(idx, size=int(0.1*len(idx)), replace=False); truth[c] = (m_, X0.loc[m_, c].values.copy()); X0.loc[m_, c] = np.nan
imp = IterativeImputer(estimator=RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42), max_iter=5, initial_strategy='median', random_state=42)
Z = pd.DataFrame(imp.fit_transform(X0), columns=shu_num, index=X0.index); out = []
for c in cols:
    m_, t_ = truth[c]; sd_ = w[c].std()
    out.append(dict(variable=c, nRMSE_RF_MICE=np.sqrt(np.mean((Z.loc[m_, c].values-t_)**2))/sd_, nRMSE_median=np.sqrt(np.mean((X0[c].median()-t_)**2))/sd_))
print(pd.DataFrame(out).round(3).to_string(index=False))

      variable  nRMSE_RF_MICE  nRMSE_median
           RSI          0.435         0.879
    Peak.Power          0.340         1.016
 Peak.Power.BM          0.469         1.139
   Jump.Height          0.385         0.895
   Body.Weight          0.537         0.815
RT.Volume.Load          0.940         1.227


## 8. How the notebook outputs map to the manuscript
* Section 2 -> corrections in Sections 3 (weeks, missingness, `#DIV/0!`, identical asymmetry columns, arithmetic identity for Total g-Load).
* Section 4 -> Table 9 (leakage-controlled TRTR/TSTR).
* Section 5 -> Tables 12-13 (sensitivity to alpha).
* Section 6 -> Tables 10-11 (distance proxies, NNDR, membership-inference AUC).

Remaining placeholders in the manuscript (CTGAN rows, seed list, repository DOI) cannot be filled from this notebook.